In [0]:
dbutils.widgets.text("start_date", "2026-05-11")
dbutils.widgets.text("end_date", "2026-05-12")


start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

bronze_adls = "abfss://bronze@dataearth.dfs.core.windows.net/"
silver_adls = "abfss://silver@dataearth.dfs.core.windows.net/"

In [0]:
from pyspark.sql.functions import col,isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

In [0]:
df = spark.read.option("multiline", True).json(f"{bronze_adls}{start_date}_earthquake_data.json")

In [0]:
df.printSchema()

root
 |-- geometry: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- id: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- alert: string (nullable = true)
 |    |-- cdi: double (nullable = true)
 |    |-- code: string (nullable = true)
 |    |-- detail: string (nullable = true)
 |    |-- dmin: double (nullable = true)
 |    |-- felt: long (nullable = true)
 |    |-- gap: long (nullable = true)
 |    |-- ids: string (nullable = true)
 |    |-- mag: double (nullable = true)
 |    |-- magType: string (nullable = true)
 |    |-- mmi: double (nullable = true)
 |    |-- net: string (nullable = true)
 |    |-- nst: long (nullable = true)
 |    |-- place: string (nullable = true)
 |    |-- rms: double (nullable = true)
 |    |-- sig: long (nullable = true)
 |    |-- sources: string (nullable = true)
 |    |-- status: string (nullable = true)
 |   

In [0]:
#Reshape the data to be more readable
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)
    

In [0]:
df

DataFrame[id: string, longitude: double, latitude: double, elevation: double, title: string, place_description: string, sig: bigint, mag: double, magType: string, time: bigint, updated: bigint]

In [0]:
#Checking for null values
df = (
    df
    .withColumn('longitude', when(isnull(col('longitude')), 0).otherwise(col('longitude')))
    .withColumn('latitude', when(isnull(col('latitude')), 0).otherwise(col('latitude')))
    .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))
)

In [0]:
#Convert 'time' and 'updated' to timestamps

df = (
    df
    .withColumn('time', (col('time')/1000).cast(TimestampType()))
    .withColumn('updated', (col('updated')/1000).cast(TimestampType()))
)


In [0]:
df.head()

Row(id='ci41463336', longitude=-115.552833333333, latitude=32.9683333333333, elevation=9.47, title='M 1.5 - 2 km WSW of Brawley, CA', place_description='2 km WSW of Brawley, CA', sig=36, mag=1.52, magType='ml', time=datetime.datetime(2026, 5, 10, 23, 57, 17, 460000), updated=datetime.datetime(2026, 5, 11, 0, 7, 42, 50000))

In [0]:
silver_output_path = f"{silver_adls}earthquake_events_silver/"

In [0]:
df.write.mode('append').parquet(silver_output_path)